# QLoRA Fine-tune — Text-to-SQL on Spider (Colab T4)

Phase 1 of the Multi-Agent Analyst System. Fine-tunes a 4-bit base model with QLoRA so it reliably turns *(schema + question)* into SQL. The exported adapter is loaded later by `src/tools/text_to_sql.py` as a live agent tool.

**Defaults (per spec):** 4-bit base, LoRA `r=16`, `alpha=32`, target *all* linear layers, LR `2e-4`, gradient checkpointing. Fits on a **free Colab T4**.

**Runtime:** set `Runtime > Change runtime type > T4 GPU` before running.

In [ ]:
# 1. Install — let Unsloth pull a mutually-compatible torch/peft/bitsandbytes/trl
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade datasets   # for load_dataset("spider")

In [ ]:
# 2. Config — single place to change knobs
BASE_MODEL    = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # 4-bit, fits T4
MAX_SEQ_LEN   = 2048
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.0
LEARNING_RATE = 2e-4
EPOCHS        = 1            # 1 epoch over Spider train is plenty to show a delta
BATCH_SIZE    = 2
GRAD_ACCUM    = 4           # effective batch 8
MAX_TRAIN     = 0           # 0 = all (~7k); set e.g. 3000 for a faster run
OUTPUT_DIR    = "outputs"
ADAPTER_DIR   = "adapter"
HF_REPO       = ""          # e.g. "your-username/llama31-spider-sql-qlora" to push

## Data — build Spider instruction dataset

Either upload `finetune/data/train.jsonl` (produced by `prepare_spider.py`) to the Colab session, or run the prep inline below. The prompt template **must match** `prepare_spider.py` and `text_to_sql.py` exactly.

In [ ]:
# 3. Build the dataset (schema-inline, Spider-derived — loads cleanly, no script)
from datasets import load_dataset

SYSTEM = (
    "You are a precise text-to-SQL engine. Given a database schema and a question, "
    "output a single valid SQLite query that answers it. Output ONLY the SQL, no prose."
)
PROMPT_TEMPLATE = (
    "{system}\n\n### Database schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
)

# Each row: question, context (CREATE TABLE schema), answer (SQL). Spider + WikiSQL
# derived; schema is inline so no fragile tables.json assembly. (The canonical
# `spider` loader is script-based and breaks on newer `datasets`.)
ds = load_dataset("b-mc2/sql-create-context", split="train")

EOS = "<|eot_id|>"  # Llama-3.1 end token
def to_text(ex):
    prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=ex["context"], question=ex["question"])
    return {"text": prompt + ex["answer"].strip() + EOS}

train_ds = ds.map(to_text, remove_columns=ds.column_names)
if MAX_TRAIN:
    train_ds = train_ds.select(range(min(MAX_TRAIN, len(train_ds))))
print(train_ds)
print(train_ds[0]["text"][:600])

In [ ]:
# 4. Load 4-bit base + attach LoRA (all linear layers)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],  # all linear
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 5. Train (QLoRA via TRL SFTTrainer)
# IMPORTANT: do NOT add a monkeypatch that renames tokenizer -> processing_class.
# Pass tokenizer=tokenizer; Unsloth maps it internally. Renaming it leaves Unsloth's
# own `tokenizer` parameter = None and crashes fix_untrained_tokens
# ('NoneType' object has no attribute 'convert_ids_to_tokens').
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

targs = dict(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to="none",
)

# Newer TRL (>=0.12) puts dataset_text_field/max_seq_length in SFTConfig; older TRL
# takes them as SFTTrainer kwargs. Try the modern path, fall back gracefully.
try:
    from trl import SFTConfig
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ_LEN, **targs),
    )
except (ImportError, TypeError):
    from transformers import TrainingArguments
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        args=TrainingArguments(**targs),
    )

trainer.train()

In [ ]:
# 6. Save the LoRA adapter (small — just the adapter, not the base)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

# Optional: push to HF Hub so HF Spaces can pull it at runtime.
if HF_REPO:
    from huggingface_hub import login
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print("Pushed to", HF_REPO)

# Then download adapter/ and drop it into the repo's finetune/adapter/, OR set
# SQL_ADAPTER_REPO=<HF_REPO> in .env.

In [ ]:
# 7. Quick sanity inference
FastLanguageModel.for_inference(model)
schema = "CREATE TABLE singer (name TEXT, country TEXT, age INT);"
q = "What are the names of singers from France, oldest first?"
prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=schema, question=q)
ids = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

## Next: measure execution accuracy (before vs after)

Download Spider's `database/` folder, then run `finetune/evaluate_sql.py` to get the **before/after exec-accuracy** numbers for the README proof-point table.

```bash
python finetune/evaluate_sql.py --spider-db-dir spider/database --adapter adapter --limit 200
```